In [1]:
# Import necessary libraries.
import asyncio
import openai
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import httpx

# Define the authentication URL and credentials.
auth = "https://api.uhg.com/oauth2/token"
scope = "https://api.uhg.com/.default"
grant_type = "client_credentials"


async def main():
    # Use an asynchronous client to make a POST request to the auth URL.
    async with httpx.AsyncClient() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": "f29277c3-edad-4a55-ae65-5d1ca69a71cd",
            "client_secret": "QVzwOqMt9S5yHCBWj0Uy19TCtjZwjfQh8Xg",
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = await client.post(auth, headers=headers, data=body, timeout=60)
        resp_json = resp.json()
        if "access_token" not in resp_json:
            print("Authentication failed:", resp_json)
            return
        access_token = resp_json["access_token"]

    # Define the deployment name, model usage type, and project ID.
    deployment_name = "gpt-5.4-mini_2026-03-17"

    # Define the Azure OpenAI endpoint and API version.
    shared_quota_endpoint = "https://api.uhg.com/api/cloud/api-management/ai-gateway-reasoning/1.0"
    azure_openai_api_version="2025-01-01-preview"

    # Initialize the OpenAI client.
    oai_client = openai.AzureOpenAI(
        azure_endpoint=shared_quota_endpoint,
        api_version=azure_openai_api_version,
        azure_deployment=deployment_name,
        azure_ad_token=access_token,
        default_headers={
            "projectId": "71be166f-06e3-4d57-9086-a547b626b9dc",
            # "x-upstream-env": "stg"
        }
    )

    # Define the messages to be processed by the model.
    messages = [{"role": "user", "content": "Hi, what is Prime number"}]

    # Request the model to process the messages.
    response = oai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=messages,
    )
    # Print the response from the model.
    print(response.model_dump_json(indent=2))


await main()

{
  "id": "chatcmpl-Dri77LHzbnKkT1MohjnOJ25nJzHyp",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "A **prime number** is a **whole number greater than 1** that has **exactly two positive divisors**:\n\n- **1**\n- **itself**\n\n### Examples\n- **2** is prime, because its only divisors are 1 and 2.\n- **3** is prime, because its only divisors are 1 and 3.\n- **5, 7, 11, 13** are also prime.\n\n### Not prime\n- **4** is not prime, because it can be divided by 1, 2, and 4.\n- **6** is not prime, because it can be divided by 1, 2, 3, and 6.\n\n### Important note\n- **2 is the only even prime number**, because every other even number is divisible by 2.\n\nIf you want, I can also explain **how to check whether a number is prime**.",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      },
      "

### Build a basic ChatBot with LangGraph(Graph API):


In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [3]:
class State(TypedDict):
    #Messages have the type "list". The "add_message" function is used to add messages to the list.It append messages to the list and returns the updated list.'
    messages: Annotated[list, add_messages]



In [4]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm = ChatGroq("llama3-8b-8192")
llm

In [ ]:
#we can initialize the chat model using the init_chat_model function from langchain.chat_models. This function takes in the model name and returns a chat model object that can be used to generate responses to user input.
llm = init_chat_model("groq:llama3-8b-8192")
llm

In [ ]:
#Node Functionality:
def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

#adding node:
graph_builder.add_node("llmchatbot",chatbot)

#adding edges:
graph_builder.add_edge(START,"llmchatbot")
graph_builder.add_edge("llmchatbot",END)

#complete graph:
graph = graph_builder.compile()

In [1]:
#Visualize the graph:
from IPython.display import display, Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
#Call the graph with a prompt:
response = graph.invoke({"messages":"What is the capital of France?"})

In [ ]:
response["messages"][-1].content

In [ ]:
for event in graph.stream({"messages":"What is the capital of France?"}):
    for value in event.values():
        print(value["messages"][-1].content)

### Chatbot with Tools:


In [ ]:
from langchain_tavily import TavilySearch
tool = TavilySearch(max_results=2)
tool.invoke("What is the capital of France?")


In [ ]:
#custom function:
def multiply(a: int, b: int) -> int:
    """multiply a and b

    Args:
        a (int): The first number to multiply.
        b (int): The second number to multiply.
    Returns:
        int: The product of a and b.
    """
    return a * b

In [ ]:
tools = [tool, multiply]

In [ ]:
llm_with_tool = llm.bind_tools(tools)

In [ ]:
#StateGraph:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

#Node Definition:
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

#Graph Builder:
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

#Add Edges:
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message(result) from assistant is a tool call, then tools_condition routes to tools.
    #if the message (result) from assistant is not a tool call, then tools_condition routes to END.
    tools_condition,
)
builder.add_edge("tools",END)

#compile the graph:
graph = builder.compile()

#View the graph:
from IPython.display import display, Image
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
response=graph.invoke({"messages":"What is the recent AI news?"})

In [ ]:
response["messages"][-1].content

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
#output:
response=graph.invoke({"messages":"What is  2 multiply with 3?"})
for m in response["messages"]:
    m.pretty_print()

In [ ]:
#output:
response=graph.invoke({"messages":"What is  2 multiply with 3 and then multiply with 4?"})
for m in response["messages"]:
    m.pretty_print()

### ReAct Agent Architecture:

In [ ]:
#StateGraph:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

#Node Definition:
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

#Graph Builder:
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

#Add Edges:
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message(result) from assistant is a tool call, then tools_condition routes to tools.
    #if the message (result) from assistant is not a tool call, then tools_condition routes to END.
    tools_condition,
)
builder.add_edge("tools","tool_calling_llm")

#compile the graph:
graph = builder.compile()

#View the graph:
from IPython.display import display, Image
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
#output:
response=graph.invoke({"messages":"Give me the recent AI news and multiply 3 by 4?"})
for m in response["messages"]:
    m.pretty_print()

### Adding memory in Agentic Graph:

In [ ]:
#StateGraph:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

#Node Definition:
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

#Graph Builder:
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

#Add Edges:
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message(result) from assistant is a tool call, then tools_condition routes to tools.
    #if the message (result) from assistant is not a tool call, then tools_condition routes to END.
    tools_condition,
)
builder.add_edge("tools","tool_calling_llm")

#compile the graph:(added checkpointer to save the state of the graph after each node execution. This allows us to keep track of the conversation history and the results of tool calls, which can be useful for debugging and analysis.)
graph = builder.compile(checkpointer=memory)

#View the graph:
from IPython.display import display, Image
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
#added configurable thread_id to keep track of the conversation thread in the memory saver. This allows us to associate the conversation history and tool call results with a specific thread, which can be
config ={"configurable":{"thread_id":"1"}}
response=graph.invoke({"messages":"Hi my name is Shamser"},config=config)
response

In [ ]:
response["messages"][-1].content

In [ ]:

#After adding thread id , llm is able to remember the previous conversation and respond accordingly. When we ask "Hey ! What is my name", it should be able to recall that we said "Hi my name is Shamser" in the previous message and respond with "Your name is Shamser".
response=graph.invoke({"messages":"Hey ! What is my name"},config=config)
print(response["messages"][-1].content)

### Streaming:
- Methods: stream() and astream()
-- these methods are sync and async methods for streaming back results.
> Additional parameters in streaming modes for the graph state:
 - values: This streams the full state of the graph after each node is called.
 - updates: This stream updates to the state of the graph after each node is called.


In [ ]:
#Intiate memory saver:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [ ]:
#Node definition:
def superbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
#Create a graph with superbot node and memory saver as checkpointer:
graph = StateGraph(State)

#Node:
graph.add_node("SuperBot", superbot)

#EDges:
graph.add_edge(START,"SuperBot")
graph.add_edge("SuperBot",END)

graph_builder = graph.compile(checkpointer=memory)

#Display the graph:
from IPython.display import display, Image
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
#Invoke the graph:
config ={"configurable":{"thread_id":"1"}}
graph_builder.invoke({"messages":"Hi, I am Shamser and I like Football"},config=config)

In [ ]:
#Create a thread: we are using stream method to stream the results as they become available. We are also passing the thread_id in the config to keep track of the conversation thread in the memory saver.
config ={"configurable":{"thread_id":"2"}}

#stream mode = updates, which means that the graph will stream updates to the user as they become available. This allows for a more interactive experience, as the user can see the results of their input in real-time.
for chunk in graph_builder.stream({"messages":"Hi, I am Shamser and I like Football"},config=config,stream_mode="updates"):
    print(chunk)

In [ ]:
#stream mode = values, which means that the graph will stream the final values of the nodes to the user as they become available. This allows for a more efficient experience, as the user can see the final results of their input without having to wait for all intermediate updates.
for chunk in graph_builder.stream({"messages":"Hi, I am Shamser and I like Football"},config=config,stream_mode="values"):
    print(chunk)

In [ ]:
#astream mode = messages, which means that the graph will stream the final messages of the nodes to the user as they become available. This allows for a more efficient experience, as the user can see the final messages of their input without having to wait for all intermediate updates.

config ={"configurable":{"thread_id":"3"}}
async for event in graph_builder.astream_events({"messages":["Hi, I am Shamser and I like Football"]},config=config,version="v2",):
    print(event)